# 05 — Data Types & Type Casting

Correct dtypes reduce memory by 50–90%, prevent silent bugs, and unlock performance.
This notebook covers casting, categorical, nullable types, and Pandas 2.x Arrow/nullable backends.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 200

employees = pd.DataFrame({
    'emp_id':           range(1001, 1001 + n),
    'name':             [f'Employee_{i:03d}' for i in range(n)],
    'department':       np.random.choice(['Engineering', 'Sales', 'HR', 'Marketing', 'Finance'], n),
    'salary':           np.random.randint(40000, 150000, n),
    'experience_years': np.random.randint(1, 20, n),
    'join_date':        pd.date_range('2010-01-01', periods=n, freq='W'),
    'rating':           np.random.choice([1.0, 2.0, 3.0, 4.0, 5.0], n),
    'city':             np.random.choice(['Mumbai', 'Delhi', 'Bangalore', 'Pune', 'Chennai'], n),
    'active':           np.random.choice([True, False], n, p=[0.85, 0.15])
})

print(employees.dtypes)

## 1. Default dtypes and Memory Cost

In [ ]:
dtype_sizes = {
    'int8':    1,  'int16':   2,  'int32':  4,  'int64':   8,
    'float32': 4,  'float64': 8,
    'bool':    1,  'object':  'variable (~50+ bytes/element)'
}
for dtype, size in dtype_sizes.items():
    print(f"  {dtype:<12} {size} byte(s) per element")

In [ ]:
# Default Pandas uses int64 and float64 even when int8 would suffice
mem_before = employees.memory_usage(deep=True).sum()
print(f"Memory before optimization: {mem_before / 1024:.1f} KB")
print(employees.dtypes)

## 2. astype() — Basic Casting

In [ ]:
# Single column
employees['emp_id'] = employees['emp_id'].astype('int32')
employees['experience_years'] = employees['experience_years'].astype('int8')  # 1-20 fits int8
employees['rating'] = employees['rating'].astype('float32')

# Multiple columns at once with dict
employees = employees.astype({
    'salary': 'int32',
})

print(employees.dtypes)

In [ ]:
# astype() will raise on invalid conversion
bad_series = pd.Series(['100', '200', 'N/A'])

try:
    bad_series.astype('int64')
except ValueError as e:
    print(f"ValueError: {e}")

# errors='ignore' silently returns original on failure
result = bad_series.astype('int64', errors='ignore')
print(result)  # unchanged — still object

## 3. pd.to_numeric() — Safe Numeric Conversion

In [ ]:
# coerce: converts failures to NaN (most useful in practice)
messy = pd.Series(['100', '200', 'N/A', '300', 'unknown', '150'])
cleaned = pd.to_numeric(messy, errors='coerce')
print(cleaned)
print(f"dtype: {cleaned.dtype}")

In [ ]:
# downcast — automatically select smallest valid dtype
s = pd.Series([1, 2, 3, 4, 100])
print(f"Before downcast: {s.dtype}")

s_down = pd.to_numeric(s, downcast='integer')
print(f"After downcast:  {s_down.dtype}")  # int8 fits 1-100

f = pd.Series([1.0, 2.5, 3.14])
f_down = pd.to_numeric(f, downcast='float')
print(f"Float downcast:  {f_down.dtype}")  # float32

## 4. Categorical dtype — Memory Champion

In [ ]:
# Object vs Category — dramatic memory difference for low-cardinality columns
n_rows = 1_000_000
depts = np.random.choice(['Engineering', 'Sales', 'HR', 'Marketing', 'Finance'], n_rows)

obj_series = pd.Series(depts)           # object dtype
cat_series = pd.Series(depts, dtype='category')  # category dtype

obj_mem = obj_series.memory_usage(deep=True)
cat_mem = cat_series.memory_usage(deep=True)

print(f"Object dtype:   {obj_mem / 1024**2:.1f} MB")
print(f"Category dtype: {cat_mem / 1024**2:.1f} MB")
print(f"Memory saved:   {(1 - cat_mem/obj_mem):.1%}")

In [ ]:
# Categorical mechanics
dept_cat = employees['department'].astype('category')
print(f"Categories:  {dept_cat.cat.categories.tolist()}")
print(f"Codes (int): {dept_cat.cat.codes.head(10).tolist()}")
# Internally stored as small integers → memory efficient

In [ ]:
# ORDERED categorical — enables meaningful comparisons and sorting
level_order = ['Junior', 'Mid', 'Senior', 'Executive']
levels = pd.Categorical(
    ['Senior', 'Junior', 'Executive', 'Mid', 'Junior'],
    categories=level_order,
    ordered=True
)

s_levels = pd.Series(levels, name='level')
print(s_levels.sort_values())        # sorted by category order
print()
print(s_levels > 'Mid')             # comparison uses order

## 5. Categorical Operations — cat Accessor

In [ ]:
dept_cat = employees['department'].astype('category')

# add_categories — add a category that doesn't appear in data
dept_cat = dept_cat.cat.add_categories(['Legal'])
print(dept_cat.cat.categories.tolist())

# rename_categories
dept_renamed = dept_cat.cat.rename_categories({'HR': 'Human Resources'})
print(dept_renamed.cat.categories.tolist())

# remove_unused_categories
dept_eng_only = dept_cat[dept_cat == 'Engineering'].cat.remove_unused_categories()
print(f"Unused removed: {dept_eng_only.cat.categories.tolist()}")

## 6. Nullable Integer Types — pd.Int8Dtype() etc.

In [ ]:
# Problem: int64 cannot hold NaN — it upcasts to float64!
s_int = pd.Series([1, 2, None, 4])
print(f"Classic: dtype={s_int.dtype}, values={s_int.tolist()}")  # float64!

# Solution: nullable integer
s_nullable = pd.Series([1, 2, None, 4], dtype='Int64')  # capital I
print(f"Nullable: dtype={s_nullable.dtype}, values={s_nullable.tolist()}")

In [ ]:
# Nullable integer types
for dtype in ['Int8', 'Int16', 'Int32', 'Int64', 'UInt8', 'UInt32']:
    s = pd.Series([1, 2, None], dtype=dtype)
    print(f"{dtype:8s}: {s.tolist()}, nbytes={s.nbytes}")

## 7. pd.StringDtype() vs object

In [ ]:
# object dtype: Python objects (can be mixed types!)
# StringDtype: guaranteed strings + NA support + potentially arrow-backed

s_obj = pd.Series(['Alice', 'Bob', None, 'Carol'])                  # object
s_str = pd.Series(['Alice', 'Bob', None, 'Carol'], dtype='string')  # StringDtype

print(f"object dtype: {s_obj.dtype}, NA repr: {s_obj[2]}")
print(f"string dtype: {s_str.dtype}, NA repr: {s_str[2]}")

# StringDtype uses pd.NA, object uses None/np.nan
print(f"object isna: {pd.isna(s_obj[2])}")
print(f"string isna: {pd.isna(s_str[2])}")

## 8. pd.BooleanDtype() — Nullable Boolean

In [ ]:
s_bool = pd.Series([True, False, None, True], dtype='boolean')
print(s_bool)
print(f"dtype: {s_bool.dtype}")
print(f"sum: {s_bool.sum()}")  # counts True, ignores NA

## 9. Pandas 2.x — dtype_backend

In [ ]:
from io import StringIO

csv_data = """name,salary,dept,active
Alice,90000,Engineering,True
Bob,85000,Sales,True
Carol,,HR,False
Dave,110000,Engineering,True
"""

# Classic numpy backend
df_numpy = pd.read_csv(StringIO(csv_data))
print("NumPy backend:")
print(df_numpy.dtypes)

# Nullable numpy backend (Pandas 1.5+)
df_nullable = pd.read_csv(StringIO(csv_data), dtype_backend='numpy_nullable')
print("\nNullable NumPy backend:")
print(df_nullable.dtypes)

In [ ]:
# PyArrow backend (Pandas 2.0+) — often fastest for IO and string ops
try:
    df_arrow = pd.read_csv(StringIO(csv_data), dtype_backend='pyarrow')
    print("PyArrow backend:")
    print(df_arrow.dtypes)
except ImportError:
    print("pyarrow not installed. Install with: pip install pyarrow")

## 10. pd.to_datetime() — String to DateTime

In [ ]:
dates_raw = pd.Series(['2023-01-15', '2023-03-22', '2023-12-01', 'invalid', None])

# errors='coerce': invalid → NaT
dates = pd.to_datetime(dates_raw, errors='coerce')
print(dates)
print(f"dtype: {dates.dtype}")
print(f"NaT count: {dates.isna().sum()}")

In [ ]:
# Specify format explicitly — much faster for known formats
dates_dd_mm = pd.Series(['15/01/2023', '22/03/2023', '01/12/2023'])
parsed = pd.to_datetime(dates_dd_mm, format='%d/%m/%Y')
print(parsed)

## 11. convert_dtypes() — Smart Auto-Conversion

In [ ]:
# convert_dtypes() upgrades to best nullable types automatically
from io import StringIO

df_raw = pd.read_csv(StringIO("""
name,salary,dept,score
Alice,90000,Engineering,8.5
Bob,,Sales,7.2
Carol,85000,HR,
"""))

print("Before convert_dtypes:")
print(df_raw.dtypes)

df_smart = df_raw.convert_dtypes()
print("\nAfter convert_dtypes:")
print(df_smart.dtypes)

## 12. Real-World Memory Optimization

In [ ]:
def optimize_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """Minimize DataFrame memory footprint with safe dtype downcasting."""
    df = df.copy()
    
    for col in df.columns:
        col_dtype = df[col].dtype
        
        if pd.api.types.is_integer_dtype(col_dtype):
            df[col] = pd.to_numeric(df[col], downcast='integer')
        
        elif pd.api.types.is_float_dtype(col_dtype):
            df[col] = pd.to_numeric(df[col], downcast='float')
        
        elif col_dtype == object:
            n_unique = df[col].nunique()
            n_rows = len(df)
            # Convert to category if cardinality < 50% of rows
            if n_unique / n_rows < 0.5:
                df[col] = df[col].astype('category')
    
    return df


# Test on employees
mem_before = employees.memory_usage(deep=True).sum() / 1024
optimized = optimize_dtypes(employees)
mem_after = optimized.memory_usage(deep=True).sum() / 1024

print(f"Before: {mem_before:.1f} KB")
print(f"After:  {mem_after:.1f} KB")
print(f"Saved:  {(1 - mem_after/mem_before):.1%}")
print("\nOptimized dtypes:")
print(optimized.dtypes)

## 13. Interview: Why int + NaN = float?

In [ ]:
# Classic NumPy integers can't represent NaN (NaN is a float concept)
# So when you introduce None/NaN into an int column, Pandas upcasts to float64

s = pd.Series([1, 2, 3], dtype='int64')
print(f"Before NaN: {s.dtype}")

s_with_nan = s.copy()
s_with_nan[1] = np.nan  # forces upcast
print(f"After NaN:  {s_with_nan.dtype}")
print(s_with_nan)

# Fix with nullable integer
s_nullable = pd.Series([1, 2, 3], dtype='Int64')
s_nullable[1] = pd.NA  # stays Int64
print(f"\nNullable after NA: {s_nullable.dtype}")
print(s_nullable)

## 14. Quick Reference — dtype Cheatsheet

| Use Case | Recommended dtype |
|----------|------------------|
| Small integers (0-127) | `int8` |
| Large integers | `int32` or `int64` |
| Integers with NaN | `Int8` / `Int64` (nullable) |
| Decimals | `float32` or `float64` |
| Low-cardinality strings | `category` |
| Guaranteed strings | `string` (StringDtype) |
| Boolean with NaN | `boolean` (BooleanDtype) |
| Dates | `datetime64[ns]` |
| High-perf IO | `pyarrow` backend |

**Rule of thumb**: Convert `object` columns with < 50% unique values to `category`.